In [11]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import joblib

base_path = '/content/drive/MyDrive/dissertation_project/data'
processed_path = f'{base_path}/processed'
model_path = f'{base_path}/models'
new_test_path = f'{processed_path}/new_test'

print("Setup complete")
print("model_path:", model_path)

ValueError: Mountpoint must not already contain files

In [12]:
import pandas as pd
import numpy as np
import joblib

base_path = '/content/drive/MyDrive/dissertation_project/data'
processed_path = f'{base_path}/processed'
model_path = f'{base_path}/models'
new_test_path = f'{processed_path}/new_test'

print("Setup complete")
print("model_path:", model_path)

Setup complete
model_path: /content/drive/MyDrive/dissertation_project/data/models


In [13]:
try:
    print("merged_structured_encoded exists, shape:", merged_structured_encoded.shape)
except NameError:
    print("merged_structured_encoded NOT in memory — need to rebuild")

try:
    print("training_feature_cols_correct exists, length:", len(training_feature_cols_correct))
except NameError:
    print("training_feature_cols_correct NOT in memory — need to rebuild")

merged_structured_encoded NOT in memory — need to rebuild
training_feature_cols_correct NOT in memory — need to rebuild


In [14]:
# Reload the 284-patient reference and structured merge inputs
new_test_with_labels = pd.read_csv(f'{new_test_path}/new_test_with_labels.csv')
print("new_test_with_labels:", new_test_with_labels.shape)

admissions = pd.read_csv(f'{base_path}/raw/admissions.csv.gz')
patients = pd.read_csv(f'{base_path}/raw/patients.csv.gz')

for df in [admissions, patients, new_test_with_labels]:
    df['subject_id'] = df['subject_id'].astype(str)
new_test_with_labels['hadm_id'] = new_test_with_labels['hadm_id'].astype(str)
admissions['hadm_id'] = admissions['hadm_id'].astype(str)

structured_new = new_test_with_labels.merge(admissions, on=['subject_id', 'hadm_id'],

_IncompleteInputError: incomplete input (3264111767.py, line 13)

In [16]:
new_test_with_labels = pd.read_csv(f'{new_test_path}/new_test_with_labels.csv')
print("new_test_with_labels:", new_test_with_labels.shape)

admissions = pd.read_csv(f'{base_path}/raw/admissions.csv.gz')
patients = pd.read_csv(f'{base_path}/raw/patients.csv.gz')

admissions['subject_id'] = admissions['subject_id'].astype(str)
patients['subject_id'] = patients['subject_id'].astype(str)
new_test_with_labels['subject_id'] = new_test_with_labels['subject_id'].astype(str)
new_test_with_labels['hadm_id'] = new_test_with_labels['hadm_id'].astype(str)
admissions['hadm_id'] = admissions['hadm_id'].astype(str)

structured_new = new_test_with_labels.merge(admissions, on=['subject_id', 'hadm_id'], how='left', suffixes=('', '_adm'))
print("after admissions merge:", structured_new.shape)

structured_new = structured_new.merge(patients, on='subject_id', how='left', suffixes=('', '_pat'))
print("after patients merge:", structured_new.shape)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/dissertation_project/data/processed/new_test/new_test_with_labels.csv'

In [17]:
import os
print("Contents of new_test folder:")
for f in sorted(os.listdir(new_test_path)):
    print(" -", f)

Contents of new_test folder:


In [18]:
from google.colab import drive
drive.flush_and_unmount()

import time
time.sleep(3)

drive.mount('/content/drive', force_remount=True)
time.sleep(8)

import os
print("Contents of new_test folder after remount:")
for f in sorted(os.listdir(new_test_path)):
    print(" -", f)

Drive not mounted, so nothing to flush and unmount.


ValueError: Mountpoint must not already contain files

In [19]:
import shutil
import os

# Force-remove the stale mountpoint directory
if os.path.exists('/content/drive'):
    try:
        shutil.rmtree('/content/drive')
        print("Removed stale /content/drive directory")
    except Exception as e:
        print("Could not remove:", e)

from google.colab import drive
drive.mount('/content/drive')

import time
time.sleep(8)

print("\nContents of new_test folder after clean remount:")
for f in sorted(os.listdir(new_test_path)):
    print(" -", f)

Removed stale /content/drive directory
Mounted at /content/drive

Contents of new_test folder after clean remount:
 - images
 - new_independent_test_reference.csv
 - new_test_with_labels.csv
 - processed_images_new_test.npz
 - reports
 - test_download.jpg
 - vit_bert_features_new_test.npz


In [21]:
new_test_with_labels = pd.read_csv(f'{new_test_path}/new_test_with_labels.csv')
print("new_test_with_labels:", new_test_with_labels.shape)

admissions = pd.read_csv(f'{base_path}/raw/admissions.csv.gz')
patients = pd.read_csv(f'{base_path}/raw/patients.csv.gz')

admissions['subject_id'] = admissions['subject_id'].astype(str)
patients['subject_id'] = patients['subject_id'].astype(str)
new_test_with_labels['subject_id'] = new_test_with_labels['subject_id'].astype(str)
new_test_with_labels['hadm_id'] = new_test_with_labels['hadm_id'].astype(str)
admissions['hadm_id'] = admissions['hadm_id'].astype(str)

structured_new = new_test_with_labels.merge(admissions, on=['subject_id', 'hadm_id'], how='left', suffixes=('', '_adm'))
print("after admissions merge:", structured_new.shape)

structured_new = structured_new.merge(patients, on='subject_id', how='left', suffixes=('', '_pat'))
print("after patients merge:", structured_new.shape)

new_test_with_labels: (284, 24)
after admissions merge: (284, 38)
after patients merge: (284, 43)


In [22]:
lab_dict = pd.read_csv(f'{processed_path}/lab_feature_dictionary.csv')
target_hadm_ids = set(structured_new['hadm_id'].astype(str))
target_subject_ids = set(structured_new['subject_id'].astype(str))
itemid_to_canonical = dict(zip(lab_dict['itemid'].astype(str), lab_dict['canonical_name']))
relevant_itemids = set(lab_dict['itemid'].astype(str))

matching_rows = []
chunk_iter = pd.read_csv(f'{base_path}/raw/labevents.csv.gz', usecols=['subject_id', 'hadm_id', 'itemid', 'valuenum'], dtype={'subject_id': str, 'hadm_id': str, 'itemid': str}, chunksize=500000)

for chunk in chunk_iter:
    filtered = chunk[chunk['subject_id'].isin(target_subject_ids) & chunk['itemid'].isin(relevant_itemids) & chunk['valuenum'].notna()]
    if len(filtered) > 0:
        matching_rows.append(filtered)

lab_events_filtered = pd.concat(matching_rows, ignore_index=True)
lab_events_correct = lab_events_filtered[lab_events_filtered['hadm_id'].isin(target_hadm_ids)]
lab_events_correct['canonical_name'] = lab_events_correct['itemid'].map(itemid_to_canonical)
lab_pivot = lab_events_correct.groupby(['hadm_id', 'canonical_name'])['valuenum'].mean().reset_index()
lab_wide = lab_pivot.pivot(index='hadm_id', columns='canonical_name', values='valuenum').reset_index()
lab_wide = lab_wide.rename(columns={'bun': 'urea_nitrogen', 'platelets': 'platelet_count'})

print("Lab wide shape:", lab_wide.shape)

Lab wide shape: (260, 21)


/tmp/ipykernel_1336/1947870674.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lab_events_correct['canonical_name'] = lab_events_correct['itemid'].map(itemid_to_canonical)


In [23]:
needed_labs = ['glucose', 'creatinine', 'sodium', 'potassium', 'hemoglobin', 'platelet_count', 'urea_nitrogen']

structured_new['hadm_id'] = structured_new['hadm_id'].astype(str)
lab_wide['hadm_id'] = lab_wide['hadm_id'].astype(str)

merged_structured = structured_new.merge(lab_wide[['hadm_id'] + needed_labs], on='hadm_id', how='left')

merged_structured['admittime'] = pd.to_datetime(merged_structured['admittime'])
merged_structured['dischtime'] = pd.to_datetime(merged_structured['dischtime'])
merged_structured['length_of_stay_hours'] = (merged_structured['dischtime'] - merged_structured['admittime']).dt.total_seconds() / 3600

categorical_cols = ['gender', 'admission_type', 'insurance', 'marital_status', 'race']
merged_structured_encoded = pd.get_dummies(merged_structured, columns=categorical_cols, drop_first=True)

print("Encoded shape:", merged_structured_encoded.shape)

Encoded shape: (284, 80)


In [24]:
# Load the training feature column list and corrected scaler
import json
with open(f'{model_path}/structured_feature_columns.json', 'r') as f:
    training_feature_cols_correct = json.load(f)

corrected_scaler_stage1 = joblib.load(f'{model_path}/structured_scaler_stage1_reconstructed.pkl')

# Add missing one-hot columns as 0
missing_cols = [c for c in training_feature_cols_correct if c not in merged_structured_encoded.columns]
print("Missing columns (added as 0):", missing_cols)

for col in missing_cols:
    merged_structured_encoded[col] = 0

# Build aligned dataframe (study_id + 58 features, no

Missing columns (added as 0): ['insurance_Unknown', 'marital_status_Unknown', 'race_ASIAN - KOREAN', 'race_ASIAN - SOUTH EAST ASIAN', 'race_HISPANIC/LATINO - CENTRAL AMERICAN', 'race_HISPANIC/LATINO - COLUMBIAN', 'race_HISPANIC/LATINO - CUBAN', 'race_HISPANIC/LATINO - GUATEMALAN', 'race_HISPANIC/LATINO - MEXICAN', 'race_HISPANIC/LATINO - SALVADORAN', 'race_NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER', 'race_WHITE - BRAZILIAN', 'race_WHITE - EASTERN EUROPEAN']


In [26]:
for col in missing_cols:
    merged_structured_encoded[col] = 0

print("Missing columns added successfully")
print("merged_structured_encoded shape:", merged_structured_encoded.shape)

Missing columns added successfully
merged_structured_encoded shape: (284, 93)


In [27]:
aligned_structured = merged_structured_encoded[['study_id'] + training_feature_cols_correct].copy()
print("aligned_structured shape:", aligned_structured.shape)

aligned_structured shape: (284, 59)


In [28]:
aligned_structured[training_feature_cols_correct] = aligned_structured[training_feature_cols_correct].fillna(0)
print("Missing values after fill:", aligned_structured[training_feature_cols_correct].isna().sum().sum())

Missing values after fill: 0


In [29]:
aligned_structured[training_feature_cols_correct] = corrected_scaler_stage1.transform(aligned_structured[training_feature_cols_correct])

print("Scaling applied successfully")
print("Sample values:")
display(aligned_structured[['anchor_age', 'glucose', 'sodium']].head(3))

Scaling applied successfully
Sample values:


,anchor_age,glucose,sodium
0,-2.064452,-0.505631,0.350019
1,-0.420881,-0.053182,0.280344
2,0.492213,0.396309,0.361631


In [32]:
import torch
import torch.nn as nn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [35]:
class StructuredMLP(nn.Module):
    def __init__(self, input_size, num_labels, feature_size=128):
        super().__init__()
        self.feature_layer = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.classifier = nn.Linear(128, num_labels)

    def forward(self, x):
        features = self.feature_layer(x)
        logits = self.classifier(features)
        return logits, features

mlp_checkpoint = torch.load(f'{model_path}/structured_mlp.pt', map_location=device)
mlp_model = StructuredMLP(input_size=len(training_feature_cols_correct), num_labels=7)
mlp_model.load_state_dict(mlp_checkpoint['model_state_dict'])
mlp_model = mlp_model.to(device).eval()

X_structured_input = torch.tensor(aligned_structured[training_feature_cols_correct].values, dtype=torch.float32).to(device)

with torch.no_grad():
    _, structured_features = mlp_model(X_structured_input)
    structured_features = structured_features.cpu().numpy()

print("Structured features shape:", structured_features.shape)

Structured features shape: (284, 128)


In [37]:
vit_bert_data = np.load(f'{new_test_path}/vit_bert_features_new_test.npz')
saved_study_ids_order = vit_bert_data['study_ids']

# Align structured features to the SAME study_id order as ViT/BERT
structured_study_ids = aligned_structured['study_id'].astype(str).values
order_map = {sid: i for i, sid in enumerate(structured_study_ids)}
reorder_idx = [order_map[sid] for sid in saved_study_ids_order.astype(str)]

structured_features_ordered = structured_features[reorder_idx]

print("Final aligned shapes:")
print("ViT:", vit_bert_data['vit_features'].shape)
print("BERT:", vit_bert_data['text_features'].shape)
print("Structured:", structured_features_ordered.shape)

np.savez(f'{new_test_path}/all_features_ready_for_testing.npz',
         vit_features=vit_bert_data['vit_features'],
         text_features=vit_bert_data['text_features'],
         structured_features=structured_features_ordered,
         study_ids=saved_study_ids_order)

print("\n✅ Saved: all_features_ready_for_testing.npz")
print("This notebook (05b) is now complete — data is ready for testing.")

Final aligned shapes:
ViT: (284, 768)
BERT: (284, 768)
Structured: (284, 128)

✅ Saved: all_features_ready_for_testing.npz
This notebook (05b) is now complete — data is ready for testing.
